In [1]:
import os
os.chdir("..")
print(os.getcwd())

c:\Users\NASA2004\OneDrive\Documents\transformer-from-scratch


In [2]:
# the decoder

import torch
import math
import torch.nn as nn
from src.positional_encoding import PositionalEncoding
from src.attention import MultiHeadAttention
from src.encoder import PositionwiseFeedForward

class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = PositionwiseFeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_output, src_mask=None, tgt_mask=None):
        attn_output, _ = self.self_attn(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout(attn_output))

        cross_attn_output, _ = self.cross_attn(x, enc_output, enc_output, src_mask)
        x = self.norm2(x + self.dropout(cross_attn_output))

        ff_output = self.feed_forward(x)
        x = self.norm3(x + self.dropout(ff_output))

        return x

In [3]:
# verifying

d_model, num_heads, d_ff = 512, 8, 2048
decoder_layer = DecoderLayer(d_model, num_heads, d_ff)

x = torch.randn(2, 7, d_model)
enc_output = torch.randn(2, 10, d_model)

out = decoder_layer(x, enc_output)
print(out.shape)

torch.Size([2, 7, 512])


In [4]:
# full decoder stacking N layers

class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers, max_len, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_len)
        self.dropout = nn.Dropout(dropout)

        self.layers = nn.ModuleList(
            [DecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)]
        )

        self.output_projection = nn.Linear(d_model, vocab_size)

    def forward(self, x, enc_output, src_mask=None, tgt_mask=None):

        x = self.dropout(self.pos_encoding(self.embedding(x) * math.sqrt(self.d_model)))

        for layer in self.layers:
            x = layer(x, enc_output, src_mask, tgt_mask)

        return self.output_projection(x)

In [5]:
# verifying

vocab_size = 8000
d_model, num_heads, d_ff, num_layers, max_len = 512, 8, 2048, 6, 100

decoder = Decoder(vocab_size, d_model, num_heads, d_ff, num_layers, max_len)

x = torch.randint(0, vocab_size, (2, 7))
enc_output = torch.randn(2, 10, d_model)

out = decoder(x, enc_output)
print(out.shape)

torch.Size([2, 7, 8000])


In [6]:
# testing the import from src

from src.decoder import Decoder

decoder = Decoder(vocab_size=8000, d_model=512, num_heads=8, d_ff=2048, num_layers=6, max_len=100)
x = torch.randint(0, 8000, (2, 7))
enc_output = torch.randn(2, 10, 512)
out = decoder(x, enc_output)
print(out.shape)

torch.Size([2, 7, 8000])
